# Places

Analysis of places in the individual biographies.

This notebook collects all the places mentioned in the individual biographies and uses wikidata to extract associated information about them, namely current names, coordinates, and adminstrative information (present day).

Relevant to this notebook is the list of attribute types that receive place names as values.

The list is the following:

- 'nascimento'
- 'jesuita-entrada'
- 'partida'
- 'chegada'
- 'estadia'
- 'estadia-x'
- 'jesuita-votos-local'
- 'jesuita-ordenacao-padre'
- 'morte'




In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()

Define attributes types that have locations as values.

In [ ]:
attributes_with_place_names = ['nascimento','jesuita-entrada','partida','chegada','estadia','estadia-x',
                'jesuita-votos-local','jesuita-ordenacao-padre','morte']

attributes_with_wikidata = [f"{attr}@wikidata" for attr in attributes_with_place_names]
attributes_with_wikidata

Collect all the individuals with attributes related to locations. Make note of the dates of the attributes and infer dates when missing, based on last known date for a previous location attribute. Extract wikidata ids for the locations if available.



In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute
from timelink.kleio.utilities import format_timelink_date
from dehergne_util import get_linked_entity_id

# show 500 rows
pd.set_option('display.max_rows', 1550)

places_of_stay = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=attributes_with_place_names,
    column_name='place',
    db=tlnb.db,
)

# fillna with "?"
places_of_stay['place'] = places_of_stay['place'].fillna('?')

# remove rows where place == '?'
places_of_stay = places_of_stay[places_of_stay['place'] != '?']

# this sequence replaces missing dates with the value of the previous date
# followed by ">". This allows to use the information that although the date
# is unkown it has happened after a certain date
# we need the id in a column to group by it (it is currently in the index)
# TODO: maybe an option in entities_with_attribute and compute intervals using
#       post quem and ante quem dates. See https://github.com/time-link/timelink-kleio/issues/1

places_of_stay['id_col'] = places_of_stay.index

# We try to infer the date of stay from the previous date known
# create a copy of the date column to replace the '0' values with NaN
places_of_stay['place.date_inferred'] = places_of_stay['place.date'].replace('0', pd.NA)

# order by id and line to have the missing date values filled with the previous date
places_of_stay = places_of_stay.sort_values(by=['id_col', 'place.line'])
# use ffill to fill the missing values with the previous value
places_of_stay['place.date_inferred'] = places_of_stay.groupby(['id_col'])['place.date_inferred'].fillna(method='ffill')
# create a column to flag the inferred dates
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']
# reset index
places_of_stay.reset_index(inplace=True)
# if data_is_inferred set date_inferred to date_inferrred+">"
places_of_stay.loc[places_of_stay['place.date_is_inferred'], 'place.date_inferred'] = places_of_stay['place.date_inferred'] + '>'
# set missing value to ''
places_of_stay['place.date_inferred'].fillna('', inplace=True)
# restore index
places_of_stay.index = places_of_stay['id_col']

# create a new column named "wikidata_id" with the wikidata id extracted from place.comment
places_of_stay['wikidata_id'] = places_of_stay['place.comment'].apply(get_linked_entity_id, linked_data_provider='wikidata', if_missing='no wikidata')
# create a new column with the formatted date
places_of_stay['place.date_inferred.formatted'] = places_of_stay['place.date_inferred'].apply(format_timelink_date)

# places_of_stay.info()
# show results
show_only=10
cols=['place','place.attr_id','wikidata_id','place.date_inferred','place.date_inferred.formatted','groupname','name','place.type','place.comment','place.original','place.date_is_inferred','place.line','place.extra_info']
places_of_stay[places_of_stay.groupname=='n'][cols].sort_values(by=['place','place.date_inferred']).head(show_only)



### Save to Excel

In [ ]:
places_of_stay[cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/locations_names.xlsx", sheet_name='wikidata', index=True)
# another version without wikidata_id for unbiased LLM reasoning
places_of_stay[['place','place.date_inferred.formatted', 'name','place.type','place.original']].sort_values(by=['place','place.date_inferred.formatted']).to_excel(f"../inferences/locations_names_places_dates_only.xlsx", sheet_name='wikidata', index=True)

### Locations with no wikidata id

In [ ]:
places_of_stay[
    (places_of_stay['wikidata_id'] == 'no wikidata') &
    (~places_of_stay['place'].str.startswith('['))
][cols].sort_values(by=['place', 'place.date_inferred'])

#### Export to Excel all unlocated places

In [ ]:
places_of_stay[places_of_stay['wikidata_id']=='no wikidata'][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/locations_names_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

Export the list of wikidata ids for later processing by wikidata-linked-data notebook.

In [ ]:
# save to csv wikidata column, except when value is "No wikidata"
places_of_stay[places_of_stay['wikidata_id'] != 'no wikidata']['wikidata_id'].drop_duplicates().to_csv(f"../inferences/wikidata-references/locations_names_wikidata.csv", index=False)

#### Only specific place of entry, unkown places

In [ ]:
from timelink.pandas import entities_with_attribute

# Choose the place of entry
place_of_entry = 'Coimbra'

# get the ids of the places of entry
missionaries_from_place = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=['jesuita-entrada'],
    the_value=place_of_entry,
    db=tlnb.db,
)
print(f"Number of missionaries from {place_of_entry}: {len(missionaries_from_place)}")
place_of_entry_ids = missionaries_from_place.index
places_of_stay_specific = places_of_stay.loc[place_of_entry_ids]
places_of_stay_specific[places_of_stay_specific['wikidata_id'] == 'No wikidata'][cols].sort_values(by=['place','place.date_inferred'])

In [ ]:
places_of_stay_specific[places_of_stay_specific['wikidata_id'] == 'No wikidata'][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_{place_of_entry}_places_of_stay_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

### Frequency of places / Frequência dos lugares

In [ ]:
places_of_stay.info()

In [ ]:
col = 'place' # subtotal by this column

# Use pandas groupby and specify unique value count for id
df_totals = places_of_stay.groupby(col).agg({'id':'nunique',
                                                'place.date_inferred':'min',
                                                'place.date':'max'})
df_totals.info()
df_totals.sort_values('id',ascending= False).head(130)

In [ ]:
# export totals to excel
df_totals.sort_values('id',ascending= False).to_excel(f"../inferences/locations_totals.xlsx", sheet_name='totals', index=True)

### Who was at a specific place


List those who stayed at a specific place



In [ ]:
place="Coimbra"

In [ ]:
cols = ['place','name','place.type','place.date_inferred','place.obs']
show_only=100
places_of_stay[places_of_stay.place==place][cols].sort_values(by=['place.date_inferred']).head(show_only)


#### Export to markdown

In [ ]:
places_of_stay.itertuples(index=True)

In [ ]:
import os
from pandas import DataFrame
from timelink.pandas import group_attributes as person_attributes

# create if it does not exist directory ../inferences/{place_of_entry}/markdown"
directory = f"../inferences/{place_of_entry}/markdown"
if not os.path.exists(directory):
    os.makedirs(directory)


for id, name in places_of_stay[places_of_stay.place==place][['name']].sort_values('name').itertuples(index=True):
    with tlnb.db.session() as session:
        p = tlnb.db.get_person(id, session=session)
        session.add(p)
        pk:str = p.to_kleio()
        pk = f"""## {name}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {name} @ timelink
```
{pk}
```

---
Select title, right click on selected title, choose extract note
"""
        fname = f"{directory}/{id}.md"
        with open(fname, 'w', encoding='utf-8') as f:
            f.write(pk)

